# Partie 2 — Backend FastAPI

Ce notebook teste l'API du RAG avec un agent simulé : aucun appel à Ollama et aucun chargement des modèles n'est nécessaire pour ces tests. Le serveur réel réutilise `partie2/rag_runtime.py`, qui reconnecte les collections persistantes de `chroma_db/` au démarrage.

Depuis la racine du projet, installez les dépendances avec `uv sync --group dev`, puis exécutez les cellules dans l'ordre.

In [8]:
import sys
from pathlib import Path

candidats = (
    *Path.cwd().resolve().parents,
    Path.cwd().resolve(),
    *Path(sys.executable).resolve().parents,
)
racine_projet = next(
    (
        dossier
        for dossier in dict.fromkeys(candidats)
        if (dossier / "pyproject.toml").is_file()
        and (dossier / "partie2" / "api.py").is_file()
    ),
    None,
)
if racine_projet is None:
    raise FileNotFoundError(
        "Racine du projet introuvable : pyproject.toml et partie2/api.py requis."
    )
if str(racine_projet) not in sys.path:
    sys.path.insert(0, str(racine_projet))

from fastapi.testclient import TestClient

from partie2.api import create_app

SOURCE_TEST = {
    "id": "table-test",
    "contenu": "Température ambiante : 10 à 35 °C",
    "metadata": {
        "source": "Manuale_ESSICCATOIO_FR.pdf",
        "page_number": 60,
        "content_type": "table",
        "image_path": None,
    },
    "distance": 0.2,
}

class FauxAgent:
    def __init__(self, resultat=None, erreur=None):
        self.resultat = resultat or {
            "reponse": "La température admise est de 10 à 35 °C [1].",
            "sources": [SOURCE_TEST],
        }
        self.erreur = erreur

    def invoke(self, entree):
        if self.erreur:
            raise self.erreur
        assert entree["question"]
        return self.resultat

def creer_client(agent):
    app = create_app(agent_factory=lambda: agent)
    return TestClient(app)

In [9]:
# La route de santé confirme que l'agent a été initialisé.
with creer_client(FauxAgent()) as client:
    sante = client.get("/health")
    assert sante.status_code == 200
    assert sante.json() == {"status": "ok", "ready": True}
print("OK — GET /health")

OK — GET /health


In [10]:
# Réponse et citation complète (document, page, type, extrait et distance).
with creer_client(FauxAgent()) as client:
    resultat = client.post("/api/chat", json={"question": "Quelle température ?"})
    assert resultat.status_code == 200
    corps = resultat.json()
    assert "10 à 35 °C" in corps["answer"]
    assert corps["sources"] == [{
        "numero": 1,
        "document": "Manuale_ESSICCATOIO_FR.pdf",
        "page": 60,
        "type": "table",
        "contenu": "Température ambiante : 10 à 35 °C",
        "distance": 0.2,
        "image_path": None,
    }]
print("OK — POST /api/chat et format des citations")

OK — POST /api/chat et format des citations


In [11]:
# Aucune source pertinente : la réponse est conservée et la liste reste vide.
with creer_client(FauxAgent({
    "reponse": "Je n'ai pas trouvé de source suffisamment pertinente.",
    "sources": [],
})) as client:
    resultat = client.post("/api/chat", json={"question": "Question inconnue"})
    assert resultat.status_code == 200
    assert resultat.json()["sources"] == []
print("OK — réponse sans source")

OK — réponse sans source


In [12]:
# Les questions vides (même après trim) et trop longues sont refusées en 422.
with creer_client(FauxAgent()) as client:
    assert client.post("/api/chat", json={"question": "   "}).status_code == 422
    assert client.post("/api/chat", json={"question": "x" * 2001}).status_code == 422
print("OK — validation des entrées")

OK — validation des entrées


In [13]:
# Une panne du service RAG/Ollama est journalisée et exposée comme erreur 502.
with creer_client(FauxAgent(erreur=RuntimeError("Ollama indisponible"))) as client:
    resultat = client.post("/api/chat", json={"question": "Question de test"})
    assert resultat.status_code == 502
    assert "Vérifiez l'index Chroma" in resultat.json()["detail"]
print("OK — gestion des erreurs du service")

Échec du traitement de la question par le RAG.
Traceback (most recent call last):
  File "C:\Users\dell\Documents\projet_AIGenerative\partie2\api.py", line 117, in chat
    resultat = agent.invoke({"question": payload.question})
  File "C:\Users\dell\AppData\Local\Temp\ipykernel_12860\3453398684.py", line 51, in invoke
    raise self.erreur
RuntimeError: Ollama indisponible


OK — gestion des erreurs du service


## Démarrer le service réel

Dans un terminal à la racine du projet, après `uv sync --group dev`, définir `OLLAMA_MODEL` avec le nom exact renvoyé par `ollama list`. Si nécessaire, ajuster `RAG_CORS_ORIGINS` (liste séparée par des virgules), puis lancer :

```powershell
$env:OLLAMA_MODEL = "qwen2.5:3b"
$env:RAG_CORS_ORIGINS = "http://localhost:5173"
uv run uvicorn partie2.api:app --host 127.0.0.1 --port 8000
```

Au démarrage, l'API charge une fois les collections Chroma et les modèles d'embedding depuis leur cache local. Ollama doit être démarré séparément. Vérifiez `http://127.0.0.1:8000/health` et envoyez une question à `POST http://127.0.0.1:8000/api/chat` au format `{"question": "..."}`.